In [1]:
# 1: Imports
import sys
import os
modules_to_clear = ['reward_functions', 'grpo_trainer', 'grpo_trainer_unsloth']
for mod in modules_to_clear:
    if mod in sys.modules:
        del sys.modules[mod]
sys.path.insert(0, '/home/jovyan/work/kata/GRPO')
import torch
import json
import numpy as np
from datetime import datetime
from datasets import Dataset

print("=" * 60)
print("GRPO Training with Unsloth")
print("=" * 60)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# 2: Reward functions
from reward_functions import (
    reward_iso_compliance,
    reward_xml_format,
    reward_no_vague_terms,
    reward_measurability,
    reward_shall_language,
    reward_appropriate_length,
    extract_xml_answer,
)
print("✓ 6 Reward functions imported")

# 3: Model configs
CONFIG = {
    # Model
    "model_name": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    "max_seq_length": 2048,
    
    # LoRA
    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    
    # GRPO - Adjusted for stability
    "learning_rate": 3e-6,
    "kl_coef": 0.001,
    "num_generations": 4,         
    "max_completion_length": 512,
    "max_prompt_length": 512,
    
    # Batch - Conservative settings
    "per_device_train_batch_size": 1, 
    "gradient_accumulation_steps": 2,   
    
    # Training
    "max_steps": 500,
    "warmup_steps": 50,
    "logging_steps": 10,
    "save_steps": 100,            
    
    # Paths
    "data_path": "/home/jovyan/work/kata/GRPO/data/train.jsonl",
    "output_dir": "/home/jovyan/work/kata/GRPO/grpo_unsloth_output",
    
    # Compilation - DISABLED
    "torch_compile": False, 
    
    # Other
    "use_vllm": False,
    "use_wandb": False,
    "seed": 42,
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("  Configuration set (FIXED)")
print(f"   Model: {CONFIG['model_name']}")
print(f"   Max Seq Length: {CONFIG['max_seq_length']} (was 192)")
print(f"   Num Generations: {CONFIG['num_generations']} (was 8)")
print(f"   Compile: {CONFIG.get('torch_compile', False)}")


# 4: System prompt
SYSTEM_PROMPT = """You are an expert requirements engineer specializing in ISO 29148 compliance.
Transform vague, informal requirements into well-formulated, measurable, ISO 29148-compliant specifications.
Respond in the following format:
<reasoning>
Analyze the vague requirement and explain your transformation approach.
</reasoning>
<answer>
The transformed ISO 29148-compliant requirement.
</answer>
Guidelines:
1. Use "shall" for mandatory requirements
2. Include measurable criteria (time, quantity, percentage)
3. Specify the system/component as subject
4. Remove vague terms (fast, efficient, user-friendly)
5. Ensure testability and verifiability
"""
print("✓ SYSTEM_PROMPT defined")


# 5: Dataset loading
def load_dataset_for_grpo(data_path, system_prompt):
    """Load your existing train.jsonl and format for TRL GRPOTrainer."""
    data = []
    
    with open(data_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                item = json.loads(line.strip())
                vague = item.get('vague', item.get('prompt', ''))
                answer = item.get('iso_compliant', item.get('chosen', ''))
                
                if vague:
                    data.append({
                        "prompt": [
                            {"role": "system", "content": system_prompt},
                            {"role": "user", "content": f"Transform this vague requirement into an ISO 29148-compliant specification:\n\n{vague}"},
                        ],
                        "answer": answer,
                    })
            except:
                continue
    
    return Dataset.from_list(data)

dataset = load_dataset_for_grpo(CONFIG["data_path"], SYSTEM_PROMPT)
print(f"✓ Dataset loaded: {len(dataset)} examples")


# 6. Load model
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    target_modules=CONFIG["target_modules"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CONFIG["seed"],
)

print(f"✓ Model loaded: {CONFIG['model_name']}")
print(f"✓ LoRA applied: r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}")
print(f"✓ Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

GRPO Training with Unsloth
PyTorch: 2.9.1+cu128
CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Workstation Edition
VRAM: 102.0 GB
✓ 6 Reward functions imported
  Configuration set (FIXED)
   Model: unsloth/Qwen2.5-3B-Instruct-bnb-4bit
   Max Seq Length: 2048 (was 192)
   Num Generations: 4 (was 8)
   Compile: False
✓ SYSTEM_PROMPT defined
✓ Dataset loaded: 900 examples
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.3: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Workstation Edition. Num GPUs = 1. Max memory: 94.962 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red col

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.1.3 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✓ Model loaded: unsloth/Qwen2.5-3B-Instruct-bnb-4bit
✓ LoRA applied: r=32, alpha=64
✓ Trainable params: 59,867,136


In [2]:
# 5.5: Install Unsloth

!pip install torch torchvision torchaudio --upgrade
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install xformers trl peft accelerate bitsandbytes

print("✓ Installation complete! Please restart the kernel.")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-3f6pjlz2/unsloth_d1615c8eead9424591ac53d327b53b17
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-3f6pjlz2/unsloth_d1615c8eead9424591ac53d327b53b17
  Resolved https://github.com/unslothai/unsloth.git to commit d59ee86feeca4e0f63964d6fa7986a3d8d343a4c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✓ Installation complete! Please restart the kernel.


In [2]:
import sys
import os
import torch
import json
import numpy as np
from datetime import datetime
from datasets import Dataset

# Clear module cache
modules_to_clear = ['reward_functions', 'grpo_trainer', 'grpo_trainer_unsloth']
for mod in modules_to_clear:
    if mod in sys.modules:
        del sys.modules[mod]

# Add path
sys.path.insert(0, '/home/jovyan/work/kata/GRPO')

print("=" * 60)
print("GRPO Training with Unsloth")
print("=" * 60)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Import reward functions
from reward_functions import (
    reward_iso_compliance,
    reward_xml_format,
    reward_no_vague_terms,
    reward_measurability,
    reward_shall_language,
    reward_appropriate_length,
    extract_xml_answer,
)
print("✓ 6 Reward functions imported")

# Configuration
CONFIG = {
    # Model
    "model_name": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    "max_seq_length": 2048,
    
    # LoRA
    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    
    # GRPO
    "learning_rate": 3e-6,
    "kl_coef": 0.001,
    "num_generations": 4,
    "max_completion_length": 512,
    "max_prompt_length": 512,
    
    # Batch
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 2,
    
    # Training
    "max_steps": 500,
    "warmup_steps": 50,
    "logging_steps": 10,
    "save_steps": 100,
    
    # Paths
    "data_path": "/home/jovyan/work/kata/GRPO/data/train.jsonl",
    "output_dir": "/home/jovyan/work/kata/GRPO/grpo_unsloth_output",
    
    # Other
    "use_vllm": False,
    "use_wandb": False,
    "seed": 42,
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("✓ CONFIG set")

# System Prompt
SYSTEM_PROMPT = """You are an expert requirements engineer specializing in ISO 29148 compliance.
Transform vague, informal requirements into well-formulated, measurable, ISO 29148-compliant specifications.
Respond in the following format:
<reasoning>
Analyze the vague requirement and explain your transformation approach.
</reasoning>
<answer>
The transformed ISO 29148-compliant requirement.
</answer>
Guidelines:
1. Use "shall" for mandatory requirements
2. Include measurable criteria (time, quantity, percentage)
3. Specify the system/component as subject
4. Remove vague terms (fast, efficient, user-friendly)
5. Ensure testability and verifiability
"""
print("✓ SYSTEM_PROMPT defined")

# Load Dataset
def load_dataset_for_grpo(data_path, system_prompt):
    """Load your existing train.jsonl and format for TRL GRPOTrainer."""
    data = []
    
    with open(data_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                item = json.loads(line.strip())
                vague = item.get('vague', item.get('prompt', ''))
                answer = item.get('iso_compliant', item.get('chosen', ''))
                
                if vague:
                    data.append({
                        "prompt": [
                            {"role": "system", "content": system_prompt},
                            {"role": "user", "content": f"Transform this vague requirement into an ISO 29148-compliant specification:\n\n{vague}"},
                        ],
                        "answer": answer,
                    })
            except:
                continue
    
    return Dataset.from_list(data)

dataset = load_dataset_for_grpo(CONFIG["data_path"], SYSTEM_PROMPT)
print(f"✓ Dataset loaded: {len(dataset)} examples")

# Load Model with Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    target_modules=CONFIG["target_modules"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CONFIG["seed"],
)

print(f"✓ Model loaded: {CONFIG['model_name']}")
print(f"✓ LoRA applied: r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}")
print(f"✓ Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

print("\n" + "=" * 60)
print("SETUP COMPLETE - Ready for Training!")
print("=" * 60)

GRPO Training with Unsloth
PyTorch: 2.9.1+cu128
CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Workstation Edition
VRAM: 102.0 GB
✓ 6 Reward functions imported
✓ CONFIG set
✓ SYSTEM_PROMPT defined
✓ Dataset loaded: 900 examples
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.3: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Workstation Edition. Num GPUs = 1. Max memory: 94.962 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.1.3 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✓ Model loaded: unsloth/Qwen2.5-3B-Instruct-bnb-4bit
✓ LoRA applied: r=32, alpha=64
✓ Trainable params: 59,867,136

SETUP COMPLETE - Ready for Training!


In [7]:
!pip install torchvision torchaudio

In [3]:
# 7: Configure GRPO Trainer
from trl import GRPOConfig

training_args = GRPOConfig(
    output_dir=CONFIG["output_dir"],
    
    # Learning
    learning_rate=CONFIG["learning_rate"],
    
    # GRPO
    num_generations=CONFIG["num_generations"],
    max_completion_length=CONFIG["max_completion_length"],
    max_prompt_length=CONFIG["max_prompt_length"],
    
    # Batch
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    
    # Schedule
    max_steps=CONFIG["max_steps"],
    warmup_steps=CONFIG["warmup_steps"],
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    
    # GRPO specific
    beta=CONFIG["kl_coef"],
    use_vllm=CONFIG["use_vllm"],
    
    # Precision
    bf16=True,
    
    # Logging
    report_to="none",
    
    # Misc
    seed=CONFIG["seed"],
    remove_unused_columns=False,
)

print("✓ GRPOConfig created")

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4
✓ GRPOConfig created


In [1]:
# 7.5: Robust Reward Wrapper

def wrap_reward_for_grpo(reward_func, tokenizer):
    """
    Handle GRPO's actual format: List[Dict[str, Tensor]]
    """
    def grpo_compatible_reward(prompts, completions, completion_ids, **kwargs):
        rewards = []
        
        # Debug first call
        if not hasattr(grpo_compatible_reward, '_debugged'):
            grpo_compatible_reward._debugged = True
            if completions:
                print(f"\n Completions type: {type(completions)}")
                print(f" Completions[0] type: {type(completions[0])}")
                if isinstance(completions[0], dict):
                    print(f" Dict keys: {list(completions[0].keys())}\n")
        
        for idx, completion in enumerate(completions):
            try:
                # Extract text from completion
                if isinstance(completion, str):
                    text = completion
                elif isinstance(completion, dict):
                    # Try to find token IDs in the dict
                    token_ids = None
                    for key in ['input_ids', 'token_ids', 'ids', 'tokens']:
                        if key in completion:
                            token_ids = completion[key]
                            break
                    
                    if token_ids is None:
                        # If dict doesn't have expected keys, use completion_ids
                        if completion_ids and idx < len(completion_ids):
                            token_ids = completion_ids[idx]
                        else:
                            print(f"No token IDs found in dict with keys: {completion.keys()}")
                            rewards.append(0.0)
                            continue
                    
                    # Convert tensor to list if needed
                    if hasattr(token_ids, 'tolist'):
                        token_ids = token_ids.tolist()
                    elif isinstance(token_ids, dict):
                        # Sometimes it's nested
                        token_ids = next(iter(token_ids.values()))
                        if hasattr(token_ids, 'tolist'):
                            token_ids = token_ids.tolist()
                    
                    # Decode
                    text = tokenizer.decode(token_ids, skip_special_tokens=True)
                
                elif isinstance(completion, (list, torch.Tensor)):
                    if hasattr(completion, 'tolist'):
                        completion = completion.tolist()
                    text = tokenizer.decode(completion, skip_special_tokens=True)
                else:
                    print(f"Unknown type: {type(completion)}")
                    rewards.append(0.0)
                    continue
                
                # Calculate reward
                reward = reward_func(text)
                rewards.append(float(reward))
                
            except Exception as e:
                print(f"Error idx {idx}: {e}")
                rewards.append(0.0)
        
        return rewards
    
    return grpo_compatible_reward

# Wrap functions
wrapped_reward_funcs = [
    wrap_reward_for_grpo(reward_iso_compliance, tokenizer),
    wrap_reward_for_grpo(reward_xml_format, tokenizer),
    wrap_reward_for_grpo(reward_no_vague_terms, tokenizer),
    wrap_reward_for_grpo(reward_measurability, tokenizer),
]

print(f"✓ Wrapped {len(wrapped_reward_funcs)} reward functions (robust)")

NameError: name 'reward_iso_compliance' is not defined

In [7]:
# 8: Create Trainer and Start Training
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    reward_funcs=wrapped_reward_funcs,
)

print(f"✓ Trainer created with {len(wrapped_reward_funcs)} reward functions")
print("\n" + "=" * 60)
print("Starting Training...")
print("NOTE: Rewards may take 150-300 steps to improve!")
print("=" * 60 + "\n")


trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


✓ Trainer created with 4 reward functions

Starting Training...
NOTE: Rewards may take 150-300 steps to improve!



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 900 | Num Epochs = 2 | Total steps = 500
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)


Unsloth: Will smartly offload gradients to save VRAM!
⚠️  Error on completion 0: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 1: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 2: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 3: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 4: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 5: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 6: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 7: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 0: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 1: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on com

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / grpo_compatible_reward / mean,rewards / grpo_compatible_reward / std,rewards / grpo_compatible_reward / mean,rewards / grpo_compatible_reward / std,rewards / grpo_compatible_reward / mean,rewards / grpo_compatible_reward / std,rewards / grpo_compatible_reward / mean,rewards / grpo_compatible_reward / std


⚠️  Error on completion 0: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 1: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 2: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 3: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 4: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 5: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 6: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 7: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 0: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 1: argument 'ids': 'dict' object cannot be interpreted as an integer
⚠️  Error on completion 2: argument 'ids': 'dict' object cannot be int

KeyboardInterrupt: 

In [11]:
# 9: Save Model
print("\n" + "=" * 60)
print("Training Complete! Saving...")
print("=" * 60)

# Save
model.save_pretrained(f"{CONFIG['output_dir']}/final_model")
tokenizer.save_pretrained(f"{CONFIG['output_dir']}/final_model")

# Save LoRA
try:
    model.save_lora(f"{CONFIG['output_dir']}/grpo_lora")
    print(f"LoRA saved to {CONFIG['output_dir']}/grpo_lora")
except:
    pass

print(f"
# CELL 10: Test the Trained Model
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

test_requirements = [
    "The system should be fast.",
    "Users need good security.",
    "Das System soll benutzerfreundlich sein.",
]

for req in test_requirements:
    print(f"\n{'='*50}")
    print(f"Input: {req}")
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Transform this vague requirement:\n\n{req}"},
    ]
    
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.8, do_sample=True)
    
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"\nOutput:\n{response[:400]}")
    
    # Score
    completion = [[{"content": response}]]
    print(f"\nScores: ISO={reward_iso_compliance(completion)[0]:.2f}, Format={reward_xml_format(completion)[0]:.2f}")Model saved to {CONFIG['output_dir']}/final_model")


Training Complete! Saving...


NameError: name 'model' is not defined

In [ ]:
# 10: Test the Trained Model
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

test_requirements = [
    "The system should be fast.",
    "Users need good security.",
    "Das System soll benutzerfreundlich sein.",
]

for req in test_requirements:
    print(f"\n{'='*50}")
    print(f"Input: {req}")
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Transform this vague requirement:\n\n{req}"},
    ]
    
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.8, do_sample=True)
    
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"\nOutput:\n{response[:400]}")
    
    # Score
    completion = [[{"content": response}]]
    print(f"\nScores: ISO={reward_iso_compliance(completion)[0]:.2f}, Format={reward_xml_format(completion)[0]:.2f}")

In [6]:
"""
GRPO Training Script - Optimized Version
Path: /home/jovyan/work/kata/GRPO/train_grpo.py
"""

# MODULE RELOAD
import sys
import importlib

# Remove cached modules
if 'grpo_trainer' in sys.modules:
    del sys.modules['grpo_trainer']
if 'reward_functions' in sys.modules:
    del sys.modules['reward_functions']


import torch
import json
import numpy as np
from pathlib import Path
import wandb
from datetime import datetime
import matplotlib.pyplot as plt
from typing import Dict, List
from tqdm import tqdm

# CORRECTED: Import from grpo_trainer.py (the optimized version)
from grpo_trainer import GRPOTrainer, RequirementsDataset


def parse_chat_format(line: str) -> dict:
    """
    Parse chat template format to extract vague and ISO-compliant requirements.
    
    Args:
        line: Chat-formatted line with user prompt and assistant response
        
    Returns:
        Dict with 'vague' and 'iso_compliant' fields
    """
    try:
        # Find the user message part
        user_start = line.find("from:")
        user_end = line.find("<|eot_id|>")
        
        if user_start != -1 and user_end != -1:
            vague = line[user_start + 5:user_end].strip()
        else:
            # Fallback: look for text between user header and eot
            user_header = "<|start_header_id|>user<|end_header_id|>"
            if user_header in line:
                parts = line.split(user_header)[1].split("<|eot_id|>")[0]
                # Remove "Generate a measurable..." prefix if present
                if "from:" in parts:
                    vague = parts.split("from:")[-1].strip()
                else:
                    vague = parts.strip()
            else:
                vague = ""
        
        # Extract the ISO-compliant requirement (assistant response)
        assistant_header = "<|start_header_id|>assistant<|end_header_id|>"
        if assistant_header in line:
            iso_part = line.split(assistant_header)[1].split("<|eot_id|>")[0].strip()
        else:
            iso_part = ""
        
        return {
            'vague': vague,
            'iso_compliant': iso_part
        }
    except Exception as e:
        print(f"Warning: Failed to parse line: {e}")
        return {'vague': '', 'iso_compliant': ''}


def load_and_split_data(
    data_path: str,
    train_size: int = 900,
    val_size: int = 50,
    test_size: int = 50,
    seed: int = 42
):
    """
    Load and split dataset into train/val/test.
    Handles both JSON and chat template formats.
    
    Args:
        data_path: Path to JSONL file
        train_size: Number of training samples
        val_size: Number of validation samples  
        test_size: Number of test samples
        seed: Random seed
        
    Returns:
        Tuple of (train_data, val_data, test_data)
    """
    # Load all data
    all_data = []
    with open(data_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            # Try to parse as JSON first
            try:
                item = json.loads(line)
                # Check if it's already in the right format
                if 'vague' not in item and 'prompt' not in item:
                    # Might be a chat template string stored as JSON
                    if isinstance(item, dict) and len(item) == 1:
                        # Single value, might be the chat string
                        chat_str = list(item.values())[0]
                        item = parse_chat_format(chat_str)
                    else:
                        # Unknown format, skip
                        continue
                elif 'prompt' in item:
                    # Rename 'prompt' to 'vague' if needed
                    item['vague'] = item.get('prompt', '')
            except json.JSONDecodeError:
                # Not JSON, treat as raw chat template
                item = parse_chat_format(line)
            
            if item.get('vague'):  # Only add if we successfully extracted vague requirement
                all_data.append(item)
    
    print(f"Loaded {len(all_data)} valid requirement pairs")
    
    # Shuffle with seed
    np.random.seed(seed)
    indices = np.random.permutation(len(all_data))
    
    # Split
    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:train_size + val_size + test_size]
    
    train_data = [all_data[i] for i in train_indices]
    val_data = [all_data[i] for i in val_indices]
    test_data = [all_data[i] for i in test_indices]
    
    return train_data, val_data, test_data


def save_data_splits(train_data, val_data, test_data, output_dir):
    """Save data splits to separate files."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    with open(output_dir / 'train.jsonl', 'w', encoding='utf-8') as f:
        for item in train_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    
    with open(output_dir / 'val.jsonl', 'w', encoding='utf-8') as f:
        for item in val_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    
    with open(output_dir / 'test.jsonl', 'w', encoding='utf-8') as f:
        for item in test_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    
    print(f"Saved data splits to {output_dir}")
    print(f"  Train: {len(train_data)} samples")
    print(f"  Val: {len(val_data)} samples")
    print(f"  Test: {len(test_data)} samples")


def plot_training_curves(
    train_losses: List[float],
    val_losses: List[float],
    val_steps: List[int],
    output_path: str
):
    """Plot training and validation loss curves."""
    plt.figure(figsize=(12, 6))
    
    # Plot training loss
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Training Loss', alpha=0.7)
    plt.xlabel('Training Step')
    plt.ylabel('Loss')
    plt.title('Training Loss over Time')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Plot validation loss
    plt.subplot(1, 2, 2)
    plt.plot(val_steps, val_losses, 'o-', label='Validation Loss', color='orange')
    plt.xlabel('Training Step')
    plt.ylabel('Loss')
    plt.title('Validation Loss over Time')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Saved training curves to {output_path}")


def compute_validation_loss(
    trainer: GRPOTrainer,
    val_dataset: RequirementsDataset,
    num_samples: int = 50
) -> float:
    """
    Compute validation loss.
    
    For GRPO, we compute the average reward as a proxy for validation loss.
    Lower validation loss = higher reward.
    
    Args:
        trainer: GRPO trainer instance
        val_dataset: Validation dataset
        num_samples: Number of samples to evaluate
        
    Returns:
        Validation loss (negative mean reward)
    """
    eval_metrics = trainer.evaluate(val_dataset, num_samples=num_samples)
    # Return negative reward as "loss" for monitoring overfitting
    validation_loss = -eval_metrics['eval/reward_mean']
    return validation_loss


def train_grpo(
    model_name: str = "Qwen/Qwen2.5-3B-Instruct",
    data_path: str = "/home/jovyan/work/kata/random_data_1000_req_vague_and_neutral.jsonl",
    output_dir: str = "/home/jovyan/work/kata/GRPO",
    # Hyperparameters from paper
    learning_rate: float = 3e-6,
    kl_coef: float = 0.001,
    temperature: float = 1.0,
    group_size: int = 16,
    questions_per_step: int = 42,
    training_steps: int = 10400,
    ref_update_freq: int = 400,
    # Monitoring
    eval_freq: int = 100,
    save_freq: int = 500,
    # Other
    use_wandb: bool = False,
    wandb_project: str = "grpo-requirements",
    seed: int = 42
):
    """
    Main GRPO training function.
    
    Args:
        model_name: Base model to fine-tune
        data_path: Path to training data JSONL
        output_dir: Directory for outputs and checkpoints
        learning_rate: Learning rate (3e-6 from paper)
        kl_coef: KL coefficient (0.001 from paper)
        temperature: Sampling temperature (1.0 from paper)
        group_size: Group sampling outputs per requirement (16 from paper)
        questions_per_step: Questions per step (42 from paper)
        training_steps: Total training steps (10,400 for ~1.6 epochs)
        ref_update_freq: Reference model update frequency (400 steps)
        eval_freq: Evaluation frequency
        save_freq: Checkpoint save frequency
        use_wandb: Whether to use Weights & Biases
        wandb_project: W&B project name
        seed: Random seed
    """
    # Set random seeds
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    # Create output directory
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Initialize W&B
    if use_wandb:
        run_name = f"grpo_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        wandb.init(
            project=wandb_project,
            name=run_name,
            config={
                "model_name": model_name,
                "learning_rate": learning_rate,
                "kl_coef": kl_coef,
                "temperature": temperature,
                "group_size": group_size,
                "questions_per_step": questions_per_step,
                "training_steps": training_steps,
                "ref_update_freq": ref_update_freq,
                "effective_batch_size": questions_per_step * group_size,
            }
        )
    
    # Load and split data
    print("Loading and splitting data...")
    train_data, val_data, test_data = load_and_split_data(
        data_path,
        train_size=900,
        val_size=50,
        test_size=50,
        seed=seed
    )
    
    # Save splits
    save_data_splits(train_data, val_data, test_data, output_dir / "data")
    
    # Create datasets
    print("Creating datasets...")
    train_path = output_dir / "data" / "train.jsonl"
    val_path = output_dir / "data" / "val.jsonl"
    
    # Initialize trainer (using the optimized GRPOTrainer)
    print("Initializing GRPO trainer...")
    trainer = GRPOTrainer(  # ✅ CORRECTED: This is the optimized version
        model_name=model_name,
        learning_rate=learning_rate,
        kl_coef=kl_coef,
        temperature=temperature,
        group_size=group_size,
        questions_per_step=questions_per_step,
        device="cuda" if torch.cuda.is_available() else "cpu",
        use_wandb=use_wandb,
        output_dir=str(output_dir)
    )
    
    # Resume from checkpoint if it exists
    resume_checkpoint = output_dir / "best_model"
    if resume_checkpoint.exists():
        print(f"\n{'='*60}")
        print(f"RESUMING from checkpoint: {resume_checkpoint}")
        print(f"{'='*60}\n")
        trainer.load_checkpoint(str(resume_checkpoint))
        start_step = 25
    else:
        start_step = 0
        print("Starting training from scratch (no checkpoint found)")
    
    # Load datasets
    val_dataset = RequirementsDataset(str(val_path), trainer.tokenizer)
    
    # Training loop
    print(f"\nStarting GRPO training for {training_steps} steps...")
    print(f"Effective batch size: {questions_per_step * group_size}")
    print(f"Reference model update frequency: every {ref_update_freq} steps (SKIPPED for LoRA)")
    print(f"Evaluation frequency: every {eval_freq} steps")
    
    train_losses = []
    val_losses = []
    val_steps = []
    best_val_loss = float('inf')
    
    # Calculate number of passes through training data
    num_epochs = training_steps * questions_per_step / len(train_data)
    print(f"Training for ~{num_epochs:.2f} epochs")
    
    pbar = tqdm(range(start_step, training_steps), desc="Training")
    
    for step in pbar:
        # Sample batch of requirements
        batch_indices = np.random.choice(len(train_data), size=questions_per_step, replace=False)
        batch_requirements = [
            f"Transform this vague requirement into ISO 29148 compliant: {train_data[i]['vague']}"
            for i in batch_indices
        ]
        
        # Training step
        metrics = trainer.train_step(batch_requirements)
        train_losses.append(metrics['loss/total'])
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f"{metrics['loss/total']:.4f}",
            'reward': f"{metrics['reward/mean']:.3f}",
            'pass_rate': f"{metrics['pass_rate']:.2%}"
        })
        
        # Log to W&B
        if use_wandb:
            wandb.log({
                **metrics,
                'step': step,
                'epoch': step * questions_per_step / len(train_data)
            })
        
        # Update reference model - SKIPPED for LoRA training
        if (step + 1) % ref_update_freq == 0:
            print(f"\nStep {step + 1} (reference model update skipped for LoRA)")
        
        # Validation
        if (step + 1) % eval_freq == 0:
            print(f"\nRunning validation at step {step + 1}...")
            
            # Compute validation loss
            val_loss = compute_validation_loss(trainer, val_dataset, num_samples=50)
            val_losses.append(val_loss)
            val_steps.append(step)
            
            # Compute full validation metrics
            val_metrics = trainer.evaluate(val_dataset, num_samples=50)
            
            print(f"Validation metrics:")
            print(f"  Loss: {val_loss:.4f}")
            print(f"  Reward: {val_metrics['eval/reward_mean']:.3f} ± {val_metrics['eval/reward_std']:.3f}")
            print(f"  Compliance: {val_metrics['eval/compliance_mean']:.2%}")
            print(f"  Pass rate: {val_metrics['eval/pass_rate']:.2%}")
            
            if use_wandb:
                wandb.log({
                    **{f"validation/{k}": v for k, v in val_metrics.items()},
                    'validation/loss': val_loss,
                    'step': step
                })
            
            # Check for overfitting
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                print(f"  New best validation loss! Saving checkpoint...")
                trainer.save_checkpoint(str(output_dir / "best_model"))
            else:
                print(f"  Validation loss increased (best: {best_val_loss:.4f})")
                if val_loss > best_val_loss * 1.1:
                    print(f"  WARNING: Possible overfitting detected!")
        
        # Save checkpoint
        if (step + 1) % save_freq == 0:
            checkpoint_dir = output_dir / f"checkpoint-{step + 1}"
            trainer.save_checkpoint(str(checkpoint_dir))
            print(f"\nSaved checkpoint to {checkpoint_dir}")
    
    # Final evaluation
    print("\nFinal evaluation on validation set...")
    final_val_metrics = trainer.evaluate(val_dataset, num_samples=50)
    print(f"Final validation metrics:")
    print(f"  Reward: {final_val_metrics['eval/reward_mean']:.3f} ± {final_val_metrics['eval/reward_std']:.3f}")
    print(f"  Compliance: {final_val_metrics['eval/compliance_mean']:.2%}")
    print(f"  Pass rate: {final_val_metrics['eval/pass_rate']:.2%}")
    
    # Save final model
    print("\nSaving final model...")
    trainer.save_checkpoint(str(output_dir / "final_model"))
    
    # Plot training curves
    print("\nGenerating training curves...")
    plot_training_curves(
        train_losses,
        val_losses,
        val_steps,
        str(output_dir / "training_curves.png")
    )
    
    # Save training history
    history = {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_steps': val_steps,
        'best_val_loss': best_val_loss,
        'config': {
            'model_name': model_name,
            'learning_rate': learning_rate,
            'kl_coef': kl_coef,
            'temperature': temperature,
            'group_size': group_size,
            'questions_per_step': questions_per_step,
            'training_steps': training_steps,
        }
    }
    
    with open(output_dir / 'training_history.json', 'w') as f:
        json.dump(history, f, indent=2)
    
    print(f"\nTraining complete! All outputs saved to {output_dir}")
    
    if use_wandb:
        wandb.finish()
    
    return trainer, history


if __name__ == "__main__":
    # CONFIGURATION is ADJUSTABLE
    DATA_PATH = "/home/jovyan/work/kata/random_data_1000_req_vague_and_neutral.jsonl"
    OUTPUT_DIR = "/home/jovyan/work/kata/GRPO"
    MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
    

    trainer, history = train_grpo(
        model_name=MODEL_NAME,
        data_path=DATA_PATH,
        output_dir=OUTPUT_DIR,
        learning_rate=1e-7,
        kl_coef=0.001,
        temperature=1.0,
        group_size=15,
        questions_per_step=21,
        training_steps=50,
        ref_update_freq=50,
        eval_freq=20,
        save_freq=50,
        use_wandb=False,
        seed=42
    )

ModuleNotFoundError: No module named 'grpo_trainer'